In [3]:
import httpx, json, time, re
from datetime import date
from collections import Counter

# ==========================================
# 1. CẤU HÌNH (Cập nhật lại URL/TOKEN nếu Colab restart)
# ==========================================
URL   = "https://qui-hebrew-lifestyle-awareness.trycloudflare.com"
TOKEN = "AjVT5_bB9I7Tq-DxwX7iba0M-9-ov4gq"
MODEL = "qwen2.5:14b-instruct-q4_K_M"

CATEGORIES = ["verb_tense","word_form","preposition","conjunction",
              "pronoun","comparison","subjunctive","vocabulary",
              "article","relative_clause"]
DIFFS      = ["easy","medium","hard"]

# Phân bổ 30 câu
PLAN_30 = [(CATEGORIES[i % 10], DIFFS[i % 3]) for i in range(30)]

# ==========================================
# 2. SYSTEM PROMPT & SCHEMA
# ==========================================
SYSTEM = f"""Bạn là chuyên gia ra đề TOEIC Part 5 (Incomplete Sentences).
Hôm nay là {date.today().isoformat()}.
ĐỊNH DẠNG MỖI CÂU:
- sentence: câu tiếng Anh hoàn chỉnh, đúng 1 chỗ trống ký hiệu "_____" (5 gạch dưới).
- A/B/C/D: 4 phương án. Chỉ 1 đúng. 3 phương án nhiễu PHẢI cùng từ loại với đáp án đúng (dễ nhầm).
- correct: "A" | "B" | "C" | "D".
- category: đúng 1 trong: verb_tense, word_form, preposition, conjunction, pronoun, comparison, subjunctive, vocabulary, article, relative_clause.
- difficulty: "easy" | "medium" | "hard".
- explanation: 1–2 câu TIẾNG VIỆT, nêu rõ vì sao đáp án đúng và vì sao các đáp án kia sai.

QUY TẮC CHẤT LƯỢNG:
- Bối cảnh đa dạng: văn phòng, tài chính, du lịch, nhân sự, marketing, IT, sản xuất…
- KHÔNG lặp câu, KHÔNG lặp cấu trúc. Câu sau phải khác câu trước về ngữ cảnh.
- Câu tự nhiên như đề TOEIC thật — không quá học thuật, không quá dài.
- Mỗi câu chỉ 1 đáp án đúng duy nhất. Kiểm tra lại trước khi trả.

VÍ DỤ MẪU:
{{
  "id": 99,
  "sentence": "The marketing team will _____ the new campaign next Monday.",
  "A": "launch", "B": "launched", "C": "launching", "D": "launches",
  "correct": "A", "category": "verb_tense", "difficulty": "easy",
  "explanation": "Sau 'will' dùng động từ nguyên mẫu → 'launch'. B/C/D sai thì hoặc sai dạng."
}}

CHỈ trả về JSON đúng schema. Không thêm chữ nào ngoài JSON."""

# Bỏ minItems/maxItems để schema linh hoạt với cả lúc sinh bù câu
PART5_SCHEMA = {
    "type": "object",
    "properties": {
        "questions": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id":         {"type": "integer"},
                    "sentence":   {"type": "string"},
                    "A":          {"type": "string"},
                    "B":          {"type": "string"},
                    "C":          {"type": "string"},
                    "D":          {"type": "string"},
                    "correct":    {"type": "string", "enum": ["A","B","C","D"]},
                    "category":   {"type": "string", "enum": CATEGORIES},
                    "difficulty": {"type": "string", "enum": DIFFS},
                    "explanation":{"type": "string"},
                },
                "required": ["id","sentence","A","B","C","D","correct","category","difficulty","explanation"]
            }
        }
    },
    "required": ["questions"]
}

# ==========================================
# 3. CÁC HÀM XỬ LÝ
# ==========================================
def gen_batch(client, specs, start_id, max_retry=2):
    """specs = [(category, difficulty), ...] → list questions"""
    spec_lines = "\n".join(
        f"  - id={start_id+i}: category={c}, difficulty={d}"
        for i,(c,d) in enumerate(specs)
    )
    user = (f"Sinh ĐÚNG {len(specs)} câu TOEIC Part 5 theo phân bổ sau:\n"
            f"{spec_lines}\n\n"
            f"Trả về JSON với mảng 'questions' có đúng {len(specs)} phần tử. "
            f"id phải khớp phân bổ. Không thêm/bớt câu.")

    payload = {
        "model": MODEL,
        "messages": [
            {"role":"system","content":SYSTEM},
            {"role":"user","content":user},
        ],
        "format": PART5_SCHEMA,
        "stream": False,
        "keep_alive": "30m",
        "options": {
            "num_ctx": 4096,
            "num_predict": 1500,
            "temperature": 0.6,
            "top_p": 0.9,
            "repeat_penalty": 1.1,
        },
    }

    for attempt in range(max_retry + 1):
        t0 = time.time()
        try:
            r = client.post(f"{URL}/api/chat",
                            headers={"Authorization": f"Bearer {TOKEN}"},
                            json=payload, timeout=300)
            r.raise_for_status()
            d = r.json()
            
            dt = time.time() - t0
            eval_duration = max(1e-9, d.get("eval_duration", 1) / 1e9)
            eval_count = d.get("eval_count", 0)
            speed = eval_count / eval_duration
            
            print(f"  ↳ {len(specs)} câu | {dt:5.1f}s | {speed:5.1f} tok/s | {eval_count} out-tok")
            
            # Làm sạch markdown nếu model trả về ```json ... ```
            content = d["message"]["content"].strip()
            if content.startswith("```json"):
                content = content[7:]
            if content.startswith("```"):
                content = content[3:]
            if content.endswith("```"):
                content = content[:-3]
            content = content.strip()
            
            data = json.loads(content)
            qs = data["questions"]
            
            if len(qs) == len(specs):
                return qs
            print(f"  ⚠ batch trả {len(qs)} câu, cần {len(specs)} → retry")
            
        except Exception as e:
            print(f"  ⚠ Lỗi (lần {attempt+1}): {e} → retry")
            time.sleep(2)
            
    return []

def validate(questions):
    """Lọc câu lỗi: trùng, thiếu blank, correct sai, thiếu đáp án"""
    seen, out = set(), []
    for q in questions:
        s = q.get("sentence","").strip()
        if not s or s.lower() in seen:
            continue
        if not re.search(r"_{3,}", s):        # bắt buộc có dấu _____
            continue
        if q.get("correct") not in ("A","B","C","D"):
            continue
        opts = [q.get(k,"").strip() for k in "ABCD"]
        if any(not o for o in opts) or len(set(opts)) < 4:
            continue
        seen.add(s.lower())
        out.append(q)
    return out

def generate_toeic_part5():
    batches = [PLAN_30[i:i+6] for i in range(0, 30, 6)]
    all_qs = []
    t_start = time.time()

    with httpx.Client(timeout=300) as client:
        for bi, batch in enumerate(batches, 1):
            print(f"[Batch {bi}/{len(batches)}]")
            start_id = (bi-1)*6 + 1
            
            # Sinh 6 câu
            qs = gen_batch(client, batch, start_id)
            valid_qs = validate(qs)
            
            # Nếu bị rớt câu do lỗi format, sinh bù
            retries = 0
            while len(valid_qs) < len(batch) and retries < 2:
                missing_count = len(batch) - len(valid_qs)
                print(f"  → Đang sinh bù {missing_count} câu...")
                missing_specs = batch[len(valid_qs):] 
                more = gen_batch(client, missing_specs, start_id + len(valid_qs))
                valid_qs.extend(validate(more))
                retries += 1
                
            # Đảm bảo không lấy dư câu
            all_qs.extend(valid_qs[:len(batch)])

    total = time.time() - t_start
    if all_qs:
        print(f"\n✅ Hoàn thành: {len(all_qs)}/30 câu | {total:.1f}s | {total/len(all_qs):.1f}s/câu")
        print("Phân bổ category:", dict(Counter(q["category"] for q in all_qs)))
        print("Phân bổ difficulty:", dict(Counter(q["difficulty"] for q in all_qs)))
    else:
        print("\n❌ Không sinh được câu nào. Hãy kiểm tra lại Colab / URL / Token.")
        
    return all_qs

# ==========================================
# 4. CHẠY CHƯƠNG TRÌNH
# ==========================================
if __name__ == "__main__":
    qs = generate_toeic_part5()
    if qs:
        with open("toeic_part5_30.json", "w", encoding="utf-8") as f:
            json.dump(qs, f, ensure_ascii=False, indent=2)
        print("💾 Đã lưu file: toeic_part5_30.json")

[Batch 1/5]
  ↳ 6 câu |  45.9s |  14.4 tok/s | 623 out-tok
[Batch 2/5]
  ↳ 6 câu |  46.6s |  13.6 tok/s | 623 out-tok
  → Đang sinh bù 1 câu...
  ↳ 1 câu |  14.3s |  14.4 tok/s | 193 out-tok
[Batch 3/5]
  ↳ 6 câu |  46.3s |  13.3 tok/s | 606 out-tok
[Batch 4/5]
  ↳ 6 câu |  45.1s |  14.0 tok/s | 614 out-tok
[Batch 5/5]
  ↳ 6 câu |  49.6s |  13.5 tok/s | 663 out-tok

✅ Hoàn thành: 30/30 câu | 247.8s | 8.3s/câu
Phân bổ category: {'verb_tense': 4, 'word_form': 5, 'pronoun': 3, 'comparison': 3, 'subjunctive': 3, 'vocabulary': 3, 'relative_clause': 3, 'preposition': 2, 'conjunction': 2, 'article': 2}
Phân bổ difficulty: {'easy': 10, 'medium': 11, 'hard': 9}
💾 Đã lưu file: toeic_part5_30.json
